# World Development Indicators data set

The goal of this notebook is to investigate influential outliers in the WDI data set.

In [ ]:
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor

In [ ]:
from main_iom import *

In [ ]:
sns.set_theme(style='white')

We read in the data and select a few variables. Our goal is to predict the GDP in 2022 of a country given captial, savings, population growth and consumption.

Download data from `"https://www.worldbank.org/ext/en/home"` and save as `"WDICSV.csv"`.

In [ ]:
df = pd.read_csv("WDICSV.csv")
df = df[["Country Name", "Country Code", "Indicator Name", "Indicator Code", "2022"]]

In [ ]:
df_final = df[df["Indicator Name"].isin(["GDP per capita (current US$)", 
                                         "Gross capital formation (current US$)", 
                                         "Population growth (annual %)", 
                                         "Gross savings (current US$)", 
                                         "Final consumption expenditure (current US$)"])] \
                                    .pivot(columns='Indicator Name', 
                                           index='Country Name', 
                                           values='2022') \
                                    .dropna()

In [ ]:
df_final.columns.name= None
df_final.index.name = None

We set up the data to be used in the model

In [ ]:
X = df_final.drop("GDP per capita (current US$)", axis=1)
y = df_final['GDP per capita (current US$)']

We set up a randomized grid search fro hyperparameter tuning.

In [ ]:
grid = {'learning_rate': [0.03, 0.1],
        'depth': [4, 6, 10],
        'l2_leaf_reg': [1, 3, 5, 7, 9]}

In [ ]:
mod = CatBoostRegressor(random_seed=123)

result = mod.randomized_search(grid, X=X, y=y, n_iter=10)

We verify the fit of the model using cross validation.

In [ ]:
pd.DataFrame(result['cv_results']).loc[19]

The final parameters chosen.

In [ ]:
result['params']

We apply the `catboost` regressor and calulate the SHAP values and residuals.

In [ ]:
explainer = shap.TreeExplainer(mod, X)
shap_values = explainer.shap_values(X)

In [ ]:
resid = y.to_numpy() - mod.predict(X)

We test if the normality assumption is valid for the SHAP values and residuals.

In [ ]:
print(f"SHAP values: {multivariate_normality(shap_values).pval:.4f}")
print(f"Residuals: {stats.shapiro(resid.reshape(-1)).pvalue:.4f}")

They reject the test for normality.

In [ ]:
iom_cat = InfluentialOutlierMetric(shap_values, resid,
                                    6, 2, 6, 
                                    1, 1, 2, 
                                    lambdas=np.concatenate([[0], np.exp(np.linspace(-5, 5, 50))]),
                                    lambdas_resid=np.concatenate([[0], np.exp(np.linspace(-5, 5, 50))]),
                                    epoch=500, epoch_resid=500)

In [ ]:
# Tune for lambda
iom_cat.find_best_lambda(alpha=0.05)
iom_cat.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_cat.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_cat = iom_cat.IOM()

Put it all together.

In [ ]:
X_df = pd.DataFrame(X, index=X.index, columns=X.columns)
X_df[["Final consumption expenditure (current US$B)", 
            "Gross capital formation (current US$B)",
            "Gross savings (current US$B)"]] = X_df[["Final consumption expenditure (current US$)", 
                                                          "Gross capital formation (current US$)",	
                                                          "Gross savings (current US$)"]]*1e-9

X_df = X_df.drop(columns=["Final consumption expenditure (current US$)", 
                                "Gross capital formation (current US$)",	
                                "Gross savings (current US$)"])

In [ ]:
iom_cat.IOM_.index = X.index
X_df["Outlier"] = np.where(iom_cat.IOM_ > iom_cat.i_star[1], "Influential outlier", "Inlier")

In [ ]:
Final = pd.concat([X_df,
                   pd.Series(y, name="GDP per capita (current US$)"),
                   pd.Series(iom_cat.chi_z.reshape(-1), name='SHAP leverage (transformed)', index=X.index),
                   pd.Series(iom_cat.chi_resid.reshape(-1), name='Residual squared (transformed)', index=X.index),
                   iom_cat.IOM_],
            axis=1)

In [ ]:
round(Final[Final["Outlier"]=="Influential outlier"], 2).drop(columns=["Outlier"]).sort_values("IOM", ascending=False).round(2)

In [ ]:
iom_cat.summary()